(S-logistic)=
# Logistic regression

Consider a problem where we attempt to predict a *binary* output, i.e., the output variable $Y$ can only take two possible values, say $0$ or $1$. For example, we may want to determine if a customer should be approved for a loan using various information (e.g., current amount of debt, credit score, etc.). Here, a linear regression model is likely to yield poor results. Even if we interpret the linear model as outputing a probability of approving the customer for the loan, observe that doubling the value of all predictors doubles the predicted probability (assuming the model has no intercept). This does not seem like an appropriate behavior in our context. 

Logistic regression provides a different approach to the problem, with a focus on predicting the probability that $Y$ is $0$ or $1$. 

## Bernoulli distribution and the sigmoid function

Recall that a random variable $Y$ has a Bernouilli distribution with parameter $p \in [0,1]$, denoted $Y \sim \textrm{Bernouilli}(p)$, if 

$$
P(Y = y) = \begin{cases}
p & \textrm{ if } y = 1\\
1-p & \textrm{ if } y = 0.
\end{cases}. 
$$

A more concise way to write the above is

$$
P(Y=y) = p^y(1-p)^{1-y} \qquad (y \in \{0,1\}).
$$

In **logistic regression**, we model the random variable $Y | X =x$ (i.e., $Y$ given that $X$ is equal to $x$) as a $\textrm{Bernouilli}(p)$ probability distribution, where $p$ is a function of $x$. The link between $x$ and $p(x)$ is provided by the *logistic function* (also called *sigmoid function*): 

$$
\sigma(x) = \frac{e^x}{1+e^x} = \frac{1}{1+e^{-x}}. 
$$

Observe that $\sigma(x) \to 1$ as $x \to \infty$ and $\sigma(x) \to 0$ as $x \to -\infty$. 

```{figure} images/Logistic-curve.png
---
width: 350 px
---
```

In logistic regression, we make the assumption that 

$$
P(Y = 1 | X = x) = \sigma(x^T \beta), 
$$ 
where $\beta \in \mathbb{R}^p$ is a vector of regression coefficients, and $x \in \mathbb{R}^p$ is a vector of predictors. Since the probabilities need to add up to $1$, we set $P(Y=0 | X=x) = 1-P(Y=1|X=x)$. We therefore have

\begin{align*}
P(Y=1 | X = x) &= \frac{e^{x^T \beta}}{1+e^{x^T\beta}} \\
P(Y=0 | X = x) &= 1-P(Y=1 | X = x) = \frac{1}{1+e^{x^T\beta}}.
\end{align*}

As a result, $P(Y=1 | X = x) \approx 1$ when $x^T \beta$ is large and positive, whereas $P(Y=1 | X = x) \approx 0$ when $x^T \beta$ is large and negative. 

Using basic algebra, we can find the inverse of the logistic function: for $y \in (0,1)$, 

$$
y = \sigma(x) = \frac{e^x}{1+e^x} \iff x = \log\left(\frac{y}{1-y}\right).
$$

Here "log" is the logarithm in base $e$. Define the *logit* function by 

$$
\textrm{logit}(y) = \log\left(\frac{y}{1-y}\right).
$$

Since $\textrm{logit}$ is the inverse of $\sigma$ and since $P(Y=1 | X=x) = \sigma(x^T \beta)$, we therefore have 

\begin{align*}
\textrm{logit}P(Y=1 | X=x) &= \log\left(\frac{P(Y=1 | X=x)}{1-P(Y=1 | X=x)}\right)\\
&=\log\left(\frac{P(Y=1 | X=x)}{P(Y=0 | X=x)}\right)\\ 
&= \textrm{logit}(\sigma(x^T \beta))\\ 
&= x^T\beta. 
\end{align*}

In "gambling language", the ratio 

$$
\frac{P(Y=1 | X=x)}{P(Y=0 | X=x)}
$$

is the "odds" that $Y=1$ against $Y=0$. Thus, in logistic regression, the "log-odds" of $Y=1$ depend on $x$ linearly, i.e., $\textrm{log-odds} = x^T\beta$. 

(S-estimating-regression)=
## Estimating the regression coefficients

Since in logistic regression, we are assuming we have a probability model for $Y$, it is natural to estimate the parameter $\beta$ using <a href="https://en.wikipedia.org/wiki/Maximum_likelihood_estimation" target="_blank">maximum likelihood</a>. 

```{admonition} Maximum likelihood estimation (MLE)

Suppose $X_1, X_2, \dots, X_n$ are observations of a probability distribution that depends on a parameter $\theta$. For example, $X_1, X_2, \dots, X_n$ could be the outcomes of flipping a coin. In that case, we would have $X_i \sim \textrm{Bernouilli}(p)$, where $\theta = p$ is the parameter (the probability that the coin lands on heads). The likelihood function it the probability of observing the data that was observed: 

$$
L(\theta) = P(X_1 = x_1, X_2 = x_2, \dots, X_n = x_n), 
$$

i.e., the probability that $X_1=x_1$ and $X_2 = x_2$ and $\dots$ and $X_n = x_n$, where $x_1, x_2, \dots, x_n$ is the actual data that was observed when the experiment was performed. Notice that this is a function of theta. For example, if the $X_i$s represent $n=100$ coin flips, the probability of observing $49$ heads and $51$ tails is very low when $p$ is close to $0$ or $1$, but is very high when $p \approx 1/2$. 

The idea of the maximum likelihood estimation (MLE) method is to choose $\theta$ to maximize the likelihood function, i.e., $\theta$ is the value of the parameter that makes it the most likely to observe the data that was observed. 

In the case of the coin flip, it is not hard to show that the MLE for $p$ is the ratio of the number of heads observed by the number of coin flips. This is, of course, very natural. For other probability distributions where an estimator may not be obvious to guess though, the MLE approach provides a rigorous and systematic way to estimate the parameters of a probability model. 

```

### Digression: the MLE for the Bernouilli distribution

Recall that if $Y \sim \textrm{Bernouilli}(p)$, then $P(Y=y) = p^y(1-p)^{1-y}$ for $y=0,1$. Hence, if $Y_1, \dots, Y_n \sim \textrm{Bernouilli}(p)$ are independent and identically distributed, then the associated likelihood function is 

$$
L(p) = P(Y_1 = y_1, \dots, Y_n = y_n) = \prod_{i=1}^n p^{y_i}(1-p)^{1-y_i} = p^{\sum_{i=1}^n y_i} (1-p)^{n-\sum_{i=1}^n y_i}. 
$$

Observe that $L(p)$ and $l(p) := \log L(p)$ are maximized at the same point since $\log$ is an increasing function. The function $l(p)$ is called the *log-likelihood* function. When working out the MLE, it is usually easier to maximze $l(p)$ as logarithms convert products into sums. Here, we have 

$$
l(p) = \left(\sum_{i=1}^n y_i\right) \log p + \left(n-\sum_{i=1}^n y_i \right) \log (1-p). 
$$

Computing the derivative $l'(p)$ and setting it to zero, it is not hard to show that $l(p)$, and thus $L(p)$, is maximized at 

$$
\widehat{p}_\textrm{MLE} = \frac{1}{n} \sum_{i=1}^n y_i. 
$$

In other words, in the coin flip problem, the MLE for the probability of getting heads is the number of heads obtained divided by the number of coin flips, as expected.

### Back to logistic regression

To estimate the regression coefficients $\beta$ in logistic regression, we can use a similar approach as above. Here, however, $p$ depends on $\beta$. For each observation $x_i$, we have 

$$
p = p(x_i, \beta) = \frac{e^{x_i^T \beta}}{1+e^{x_i^T \beta}}. 
$$

The likelihood function for $\beta$ is thus: 

$$
L(\beta) = \prod_{i=1}^n p(x_i, \beta)^{y_i} (1-p(x_i,\beta))^{1-y_i},  
$$

and the log-likelihood is: 

\begin{align*}
l(\beta) &= \sum_{i=1}^n y_i \log p(x_i, \beta) + (1-y_i) \log(1-p(x_i, \beta)) \\
&=\sum_{i=1}^n y_i (x_i^T \beta - \log(1+e^{x_i^T \beta})) - (1-y_i) \log(1+e^{x_i^T \beta}) \\
&= \sum_{i=1}^n [y_i x_i^T \beta -  \log(1+e^{x_i^T \beta})].
\end{align*}

This is the function that we need to maximize with respect to $\beta$. Taking the derivative with respect to $\beta_j$, we obtain: 

$$
\frac{\partial}{\partial \beta_j} l(\beta) = \sum_{i=1}^n \left[y_i x_{ij}  - x_{ij} \frac{e^{x_i^T \beta}}{1+e^{x_i^T \beta}}\right].
$$

Unfortunately, setting the derivatives to $0$ does not yield a nice closed-form expression for $\beta$ as before. We therefore need to use numerical methods (e.g., <a href="https://en.wikipedia.org/wiki/Newton%27s_method" target="_blank">Newton's method</a>) to maximize the log-likelihood function and obtain the MLE for $\beta$. This is implemented for us in packages such as scikit-learn.

## Logistic regression with more than two classes

So far in this chapter, we assumed the response variable is binary (i.e., it takes only two values). Logistic regression can be generalized when the response can take any of $\{1,\dots, K\}$ values. In that case, we simply replace the Bernoulli distribution by the <a href="https://en.wikipedia.org/wiki/Categorical_distribution" target="_blank">categorical distribution</a>: 

$$
P(Y=i | X=x) = p_i, \qquad 0 \leq p_i \leq 1, \quad \sum_{i=1}^K p_i = 1
$$

Each category has its own set of coefficients: 

$$
P(Y=i | X=x) = \frac{e^{x^T \beta^{(i)}}}{\sum_{i=1}^K e^{x^T \beta^{(i)}}}.
$$

The regression coefficients can be estimated using maximum likelihood as in the binary case.

## Generalized linear models

Logistic regression can be significantly generalized. When $Y \sim \textrm{Bernoulli}(p)$, observe that the expected value of $Y$ is $E(Y) = p$. Thus, in logistic regression, we assume 

1. $Y | X=x \sim \textrm{Bernouilli}(p)$.
2. $\textrm{logit}(p) = \textrm{logit}(E(Y|X=x)) = x^T\beta$. 

A whole class of models can be built using a similar construction. A *generalized linear model* (GLM) consists of 
1. A probability distribution for $Y | X=x$ from the <a href="https://en.wikipedia.org/wiki/Exponential_family" target="_blank">exponential family</a>. 
2. A *link* function $g$ such that $g(E(Y|X=x)) = x^T \beta$.

Note that the exponential family is a large family of probability distributions. It includes most of the familiar probability distributions encountered in an undergraduate probability theory or statistics course (e.g., normal, exponential, log-normal, gamma, chi-squared, beta, Dirichlet, Bernoulli, categorical, Poisson, geometric, etc.). Generalized linear models can thus be used to construct models for data coming from all kind of distributions.

## Penalized logistic regression

As for linear regression, penalized version (e.g., with $\ell_1$ or $\ell_2$ penalties) can be constructed for logistic regression in the same spirit as Ridge and LASSO regression. In that case, instead of maximizing the log-likelihood function, we can solve one of the following optimization problems:

\begin{align*}
&\min_\beta -l(\beta) + \alpha \|\beta\|_1 \\
&\min_\beta -l(\beta) + \alpha \|\beta\|_2.
\end{align*}

## Logistic regression with Scikit-learn

Constructing logistic regression models with Scikit-learn is very straightforward. 

In [ ]:
from sklearn.linear_model import LogisticRegression

The <a href="https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html" target="_blank">LogisticRegression</a> object behaves in a similar manner as the <a href="https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html" target="_blank">LinearRegression</a> object (fit, predict, score, etc.). 

To illustrate, let us construct a logistic regression model for the classical <a href="https://scikit-learn.org/stable/datasets/toy_dataset.html#iris-dataset" target="_blank">Iris dataset</a>.

In [3]:
from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

# Load iris dataset
X, y = load_iris(return_X_y=True)

# Scale the features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Create logistic regression model and fit it to data
clf = LogisticRegression(random_state=0)
clf.fit(X_scaled, y)

# Predict the labels and measure the model fit
clf.predict(X_scaled[:2, :])
clf.predict_proba(X_scaled[:2, :])
clf.score(X_scaled, y)

0.9733333333333334

We obtain a very good training accuracy. 

```{admonition} Exercise

Repeat the above exercise with the Iris dataset, but first split the data into training and test sets. Train a logistic regression model on the training set and measure its accuracy on the test set. 

```